#  03. NLP 전처리 — 영어 리뷰

## 목적
영어 리뷰 텍스트를 정제하고 NLP 분석에 사용할 수 있는 형태로 변환한다.

## 사전 조건
- `02_eda.ipynb` 실행 완료
- `data/dave_diver.db`에 reviews 테이블 존재
- spaCy 설치: `pip install spacy`
- 영어 모델 설치: `python -m spacy download en_core_web_sm`

## 분석 순서
1. 라이브러리 & DB 연결
2. 영어 리뷰 로드 & 그룹 컬럼 생성
3. 기본 텍스트 정제 (URL, 특수문자 제거)
4. spaCy 파이프라인 최적화 설정
5. 토큰화 + 표제어 추출 (배치 처리)
6. 불용어 추가 제거
7. 정제 결과 확인 (전체 / 시기별 / 유저 그룹별)
8. DB 저장
9. DB 연결 종료

## 이 노트북의 결과물
- `cleaned_reviews_en` 테이블 (cleaned_text + 분석용 그룹 컬럼 포함)
- 시기별 / 플레이타임 구간별 / 긍정·부정별 상위 토큰 비교

---
## 1. 라이브러리 & DB 연결

In [1]:
import sqlite3
import re
import pandas as pd
import numpy as np
import spacy
from collections import Counter
from tqdm import tqdm

tqdm.pandas()

DB_PATH = '../data/dave_diver.db'
conn    = sqlite3.connect(DB_PATH)

# ── spaCy 파이프라인 최적화 로드 ──────────────────────────────────────────
# lemmatizer는 tagger + attribute_ruler가 있어야 동작
# parser, ner는 이 프로젝트에서 불필요 → 속도 최적화를 위해 제외
nlp = spacy.load(
    'en_core_web_sm',
    exclude=['parser', 'ner']  # exclude: 아예 로드하지 않음 (disable보다 빠름)
)

print(f'✅ 준비 완료')
print(f'spaCy 버전        : {spacy.__version__}')
print(f'활성 파이프라인    : {nlp.pipe_names}')

✅ 준비 완료
spaCy 버전        : 3.8.11
활성 파이프라인    : ['tok2vec', 'tagger', 'attribute_ruler', 'lemmatizer']


---
## 2. 영어 리뷰 로드 & 그룹 컬럼 생성

나중에 groupby 분석(시기별 / 유저 그룹별)을 바로 할 수 있도록
분석에 필요한 컬럼을 함께 로드해서 cleaned_reviews_en에 저장한다.

저장할 그룹 컬럼:
- `review_month` : 시기별 분석 (업데이트 전후 비교 등)
- `play_segment` : 플레이타임 5구간 (casual / regular / engaged / hardcore)
- `voted_up`     : 긍정/부정
- `received_for_free` : 무료수령 여부
- `written_during_early_access` : 얼리액세스 여부

In [2]:
df_en = pd.read_sql("""
    SELECT
        review_id,
        review_text,
        voted_up,
        playtime_at_review,
        received_for_free,
        written_during_early_access,
        review_month,
        language
    FROM reviews
    WHERE language = 'english'
      AND review_text IS NOT NULL
      AND TRIM(review_text) != ''
""", conn)

# 플레이타임 시간 단위 변환
df_en['playtime_hours'] = df_en['playtime_at_review'] / 60

# 플레이타임 5구간 생성
bins_p   = [0, 120, 600, 1800, 3000, 999999]
labels_p = ['casual(<2h)', 'regular(2-10h)', 'engaged(10-30h)', 'engaged(30-50h)', 'hardcore(50h+)']
df_en['play_segment'] = pd.cut(df_en['playtime_at_review'], bins=bins_p, labels=labels_p)

print(f'영어 리뷰 총 수            : {len(df_en):,}건')
print(f'긍정                      : {df_en["voted_up"].sum():,}건 ({df_en["voted_up"].mean()*100:.1f}%)')
print(f'부정                      : {(df_en["voted_up"]==0).sum():,}건')
print(f'무료 수령                 : {df_en["received_for_free"].sum():,}건')
print(f'얼리액세스                : {df_en["written_during_early_access"].sum():,}건')
print()
print('플레이타임 구간별 분포:')
print(df_en['play_segment'].value_counts().sort_index())

영어 리뷰 총 수            : 51,565건
긍정                      : 49,858건 (96.7%)
부정                      : 1,707건
무료 수령                 : 504건
얼리액세스                : 3,304건

플레이타임 구간별 분포:
play_segment
casual(<2h)         2069
regular(2-10h)     14107
engaged(10-30h)    17625
engaged(30-50h)    10058
hardcore(50h+)      7706
Name: count, dtype: int64


---
## 3. 기본 텍스트 정제

spaCy 처리 전에 노이즈를 제거한다.

제거 대상:
- URL (http, www)
- HTML 태그
- 영문자 이외 문자 (숫자, 특수문자)
- 연속 공백

In [3]:
def basic_clean(text: str) -> str:
    """URL, HTML, 특수문자 제거 + 소문자 변환"""
    if not isinstance(text, str):
        return ''
    text = re.sub(r'http\S+|www\.\S+', '', text)   # URL 제거
    text = re.sub(r'<[^>]+>', '', text)             # HTML 태그 제거
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)        # 영문자 + 공백만 유지
    text = re.sub(r'\s+', ' ', text).strip()        # 연속 공백 제거
    return text.lower()

df_en['text_cleaned'] = df_en['review_text'].apply(basic_clean)

# 정제 후 빈 텍스트 제거
before = len(df_en)
df_en  = df_en[df_en['text_cleaned'].str.strip() != ''].copy()
print(f'정제 후 빈 텍스트 제거: {before - len(df_en):,}건')
print(f'남은 리뷰             : {len(df_en):,}건')

# 정제 전후 비교 샘플
print('\n--- 정제 전후 비교 ---')
for _, row in df_en.sample(3, random_state=42).iterrows():
    print(f'원문  : {row["review_text"][:120]}')
    print(f'정제후: {row["text_cleaned"][:120]}')
    print()

정제 후 빈 텍스트 제거: 1,017건
남은 리뷰             : 50,548건

--- 정제 전후 비교 ---
원문  : After playing mini-game during goddess of victory: Nikke event, got me hook up with fun. 
So here I am playing real gam
정제후: after playing mini game during goddess of victory nikke event got me hook up with fun so here i am playing real game and

원문  : top 10 best games on steam
정제후: top best games on steam

원문  : simply incredible
정제후: simply incredible



---
## 4. spaCy 파이프라인 최적화 설정

Context7 spaCy 문서 기준 최적화 포인트:

| 설정 | 방법 | 이유 |
|------|------|------|
| parser, ner 제외 | `exclude=` (로드 시) | 아예 로드 안 해서 메모리 절약 |
| 배치 처리 | `nlp.pipe(batch_size=500)` | 1건씩 처리보다 훨씬 빠름 |
| 품사 필터 | NOUN, VERB, ADJ, ADV만 유지 | 의미 없는 토큰 제거 |
| 최소 길이 | 2자 이상 | 단독 알파벳 제거 |

⚠️ lemmatizer는 tagger + attribute_ruler가 필요하므로 exclude 대상에서 제외

In [4]:
# 현재 파이프라인 구성 확인
print('파이프라인 구성:')
for name, component in nlp.pipeline:
    print(f'  {name}')

# 토큰 1건 처리 테스트
test_doc = nlp('The fishing mechanics are really fun and engaging')
print('\n토큰화 + 품사 + 표제어 테스트:')
print(f'{"토큰":20s} {"품사":8s} {"표제어":20s} {"불용어"}')
print('-' * 60)
for token in test_doc:
    print(f'{token.text:20s} {token.pos_:8s} {token.lemma_:20s} {token.is_stop}')

파이프라인 구성:
  tok2vec
  tagger
  attribute_ruler
  lemmatizer

토큰화 + 품사 + 표제어 테스트:
토큰                   품사       표제어                  불용어
------------------------------------------------------------
The                  DET      the                  True
fishing              NOUN     fishing              False
mechanics            NOUN     mechanic             False
are                  AUX      be                   True
really               ADV      really               True
fun                  ADJ      fun                  False
and                  CCONJ    and                  True
engaging             VERB     engage               False


---
## 5. 토큰화 + 표제어 추출 (배치 처리)

- `nlp.pipe()` 배치 처리로 속도 최적화
- 품사 필터: NOUN, VERB, ADJ, ADV만 유지
- spaCy 기본 불용어 + 길이 2자 미만 제거

⚠️ 영어 리뷰 전체 처리 — 약 5~10분 소요

In [5]:
KEEP_POS = {'NOUN', 'VERB', 'ADJ', 'ADV'}

texts = df_en['text_cleaned'].tolist()
lemmatized = []

# nlp.pipe: 배치 처리 (Context7 권장 방식)
# batch_size=500: 메모리와 속도의 균형점
for doc in tqdm(
    nlp.pipe(texts, batch_size=500),
    total=len(texts),
    desc='spaCy 처리 중'
):
    tokens = [
        token.lemma_
        for token in doc
        if token.pos_  in KEEP_POS
        and not token.is_stop
        and not token.is_punct
        and not token.is_space
        and len(token.lemma_) >= 2
    ]
    lemmatized.append(' '.join(tokens))

df_en['cleaned_text'] = lemmatized
print('\n✅ spaCy 처리 완료')

spaCy 처리 중: 100%|██████████| 50548/50548 [01:41<00:00, 497.17it/s] 


✅ spaCy 처리 완료


---
## 6. 불용어 추가 제거

spaCy 기본 불용어 외에 게임 리뷰 특화 불용어를 추가 제거한다.
- 게임 제목 자체 (dave, diver)
- 리뷰 형식어 (recommend, review, game)
- 단독으로는 분석 가치가 낮은 단어 (good, great, bad 등)

In [6]:
CUSTOM_STOPWORDS = {
    # 게임 제목
    'dave', 'diver',
    # 리뷰 형식어
    'game', 'play', 'review', 'recommend', 'hour', 'time',
    # 단독 의미 약한 동사/형용사
    'get', 'go', 'make', 'like', 'want', 'know', 'think', 'feel',
    'good', 'great', 'bad', 'really', 'lot', 'way', 'bit', 'just',
    'thing', 'little', 'pretty', 'very', 'nice', 'well'
}

def remove_custom_stopwords(text: str) -> str:
    tokens = [t for t in text.split() if t not in CUSTOM_STOPWORDS]
    return ' '.join(tokens)

df_en['cleaned_text'] = df_en['cleaned_text'].apply(remove_custom_stopwords)

# 불용어 제거 후 빈 텍스트 처리
before = len(df_en)
df_en  = df_en[df_en['cleaned_text'].str.strip() != ''].copy()
print(f'불용어 제거 후 빈 텍스트 제거: {before - len(df_en):,}건')
print(f'최종 영어 리뷰 수           : {len(df_en):,}건')

# 최종 샘플
print('\n--- 최종 정제 결과 샘플 ---')
for _, row in df_en.sample(4, random_state=42).iterrows():
    label = '긍정' if row['voted_up'] else '부정'
    print(f'[{label} / {row["play_segment"]}]')
    print(f'  원문  : {row["review_text"][:100]}')
    print(f'  정제후: {row["cleaned_text"][:100]}')
    print()

불용어 제거 후 빈 텍스트 제거: 5,666건
최종 영어 리뷰 수           : 44,882건

--- 최종 정제 결과 샘플 ---
[긍정 / engaged(10-30h)]
  원문  : I personally can not wait for this game to fully release later today, i went as far as i could in ea
  정제후: personally wait fully release later today far early access wait day perfect cherish forever

[긍정 / casual(<2h)]
  원문  : Fishin and chippin
  정제후: chippin

[긍정 / regular(2-10h)]
  원문  : A game with charm and class
  정제후: charm class

[긍정 / engaged(30-50h)]
  원문  : This game is whimsical and fun, with surprisingly good graphics. There's little flavors of lots of d
  정제후: whimsical fun surprisingly graphic flavor different reference manage cross line wildly overdo conten



---
## 7. 정제 결과 확인

### 7-1. 전체 상위 토큰
### 7-2. 긍정 vs 부정 상위 토큰 비교
### 7-3. 시기별 상위 토큰 비교 (groupby review_month)
### 7-4. 플레이타임 구간별 상위 토큰 비교 (groupby play_segment)

In [7]:
# 7-1. 전체 상위 토큰
all_tokens = ' '.join(df_en['cleaned_text']).split()
token_freq = Counter(all_tokens)

print(f'고유 토큰 수: {len(token_freq):,}개')
print(f'전체 토큰 수: {len(all_tokens):,}개')
print()
print('전체 상위 30 토큰:')
for word, cnt in token_freq.most_common(30):
    print(f'  {word:20s} {cnt:,}')

고유 토큰 수: 18,096개
전체 토큰 수: 615,485개

전체 상위 30 토큰:
  fun                  13,554
  fish                 10,256
  love                 7,473
  story                7,053
  gameplay             5,680
  restaurant           4,978
  sushi                4,127
  dive                 3,977
  new                  3,936
  character            3,696
  ve                   3,445
  relax                3,329
  diving               3,327
  mechanic             3,262
  content              3,214
  amazing              3,212
  chill                2,965
  art                  2,878
  day                  2,861
  enjoy                2,790
  loop                 2,767
  buy                  2,693
  worth                2,400
  look                 2,295
  run                  2,293
  experience           2,284
  find                 2,253
  management           2,232
  fishing              2,156
  boss                 2,125


In [8]:
# 7-2. 긍정 vs 부정 상위 토큰 비교
for label, val in [('긍정', 1), ('부정', 0)]:
    tokens = ' '.join(df_en[df_en['voted_up'] == val]['cleaned_text']).split()
    print(f'--- {label} 리뷰 상위 20 토큰 ---')
    for w, c in Counter(tokens).most_common(20):
        print(f'  {w:20s} {c:,}')
    print()

--- 긍정 리뷰 상위 20 토큰 ---
  fun                  12,819
  fish                 9,484
  love                 7,283
  story                6,615
  gameplay             5,175
  restaurant           4,479
  sushi                3,892
  dive                 3,656
  new                  3,649
  character            3,417
  relax                3,251
  ve                   3,226
  amazing              3,157
  content              3,057
  diving               3,016
  chill                2,901
  mechanic             2,869
  art                  2,743
  day                  2,581
  enjoy                2,563

--- 부정 리뷰 상위 20 토큰 ---
  fish                 772
  fun                  735
  gameplay             505
  restaurant           499
  story                438
  mechanic             393
  quest                340
  try                  328
  boss                 322
  dive                 321
  diving               311
  find                 291
  people               289
  new                

In [9]:
# 7-3. 시기별 상위 토큰 비교 (groupby review_month)
# 주요 시점만: 정식출시(2023-06), QTE패치(2023-08), DREDGE DLC(2023-12), 고질라DLC(2024-05)
target_months = ['2023-06', '2023-08', '2023-12', '2024-05']

print('=== 시기별 상위 15 토큰 ===')
for month in target_months:
    sub = df_en[df_en['review_month'] == month]
    if len(sub) == 0:
        continue
    tokens = ' '.join(sub['cleaned_text']).split()
    top    = Counter(tokens).most_common(15)
    print(f'\n[{month}] n={len(sub):,}건')
    print(', '.join([f'{w}({c})' for w, c in top]))

=== 시기별 상위 15 토큰 ===

[2023-06] n=980건
fun(251), love(191), fish(167), restaurant(115), gameplay(109), sushi(97), dive(96), story(96), amazing(95), art(83), release(82), character(80), day(79), new(76), year(73)

[2023-08] n=2,278건
fun(697), love(497), fish(437), story(401), buy(320), gameplay(282), restaurant(272), dive(229), new(229), sushi(198), character(197), mechanic(184), diving(182), content(181), ve(172)

[2023-12] n=2,353건
fun(717), fish(389), love(340), story(334), gameplay(270), restaurant(240), new(210), dive(198), sushi(183), ve(181), character(176), mechanic(175), relax(167), diving(163), amazing(156)

[2024-05] n=880건
fun(267), fish(182), love(134), story(119), restaurant(118), gameplay(103), sushi(95), dive(85), ve(79), new(69), diving(69), find(65), day(61), amazing(59), character(58)


In [10]:
# 7-4. 플레이타임 구간별 상위 토큰 비교 (groupby play_segment)
print('=== 플레이타임 구간별 부정 리뷰 상위 15 토큰 ===')
for seg in labels_p:
    sub = df_en[(df_en['play_segment'] == seg) & (df_en['voted_up'] == 0)]
    if len(sub) < 10:
        continue
    tokens = ' '.join(sub['cleaned_text']).split()
    top    = Counter(tokens).most_common(15)
    print(f'\n[{seg}] 부정 n={len(sub):,}건')
    print(', '.join([f'{w}({c})' for w, c in top]))

=== 플레이타임 구간별 부정 리뷰 상위 15 토큰 ===

[casual(<2h)] 부정 n=367건
fun(72), fish(67), gameplay(60), button(60), restaurant(56), people(47), work(42), boring(41), refund(41), find(40), control(39), try(39), look(39), diving(38), mechanic(34)

[regular(2-10h)] 부정 n=390건
fun(150), fish(140), gameplay(116), restaurant(103), dive(81), diving(77), mechanic(76), quest(74), boring(70), story(69), loop(66), find(60), repetitive(59), try(58), people(57)

[engaged(10-30h)] 부정 n=564건
fish(299), fun(299), gameplay(205), story(185), restaurant(182), quest(179), boss(175), mechanic(171), try(149), new(133), character(129), fight(126), diving(124), find(124), day(120)

[engaged(30-50h)] 부정 n=188건
fish(150), fun(137), story(98), restaurant(95), gameplay(84), boss(79), mechanic(68), add(65), dive(61), end(59), need(59), quest(55), day(53), loop(52), new(50)

[hardcore(50h+)] 부정 n=140건
fish(116), fun(77), story(68), restaurant(63), dlc(50), upgrade(45), mechanic(44), dive(42), gameplay(40), end(39), new(38), need

---
## 8. DB 저장

정제된 텍스트와 그룹 컬럼을 `cleaned_reviews_en` 테이블로 저장.
원본 reviews 테이블은 수정하지 않는다.

저장 컬럼:
- `review_id`, `voted_up`, `review_month`, `play_segment`
- `playtime_at_review`, `received_for_free`, `written_during_early_access`
- `review_text` (원문), `cleaned_text` (정제 후)

In [11]:
save_cols = [
    'review_id', 'voted_up', 'review_month', 'play_segment',
    'playtime_at_review', 'playtime_hours',
    'received_for_free', 'written_during_early_access',
    'review_text', 'cleaned_text'
]

df_en[save_cols].to_sql(
    'cleaned_reviews_en',
    conn,
    if_exists='replace',
    index=False
)

# 저장 확인
result = pd.read_sql("SELECT COUNT(*) as total FROM cleaned_reviews_en", conn)
schema = pd.read_sql("PRAGMA table_info(cleaned_reviews_en)", conn)

print(f'✅ 저장 완료: {result["total"][0]:,}건')
print()
print('저장된 컬럼:')
print(schema[['name', 'type']].to_string(index=False))

# groupby 활용 예시
print('\n--- groupby 활용 예시 ---')
print('시기별 리뷰 수 확인:')
check = pd.read_sql("""
    SELECT review_month, COUNT(*) as cnt,
           ROUND(AVG(voted_up)*100,1) as pos_rate
    FROM cleaned_reviews_en
    GROUP BY review_month
    ORDER BY review_month
    LIMIT 10
""", conn)
print(check.to_string(index=False))

✅ 저장 완료: 44,882건

저장된 컬럼:
                       name    type
                  review_id    TEXT
                   voted_up INTEGER
               review_month    TEXT
               play_segment    TEXT
         playtime_at_review INTEGER
             playtime_hours    REAL
          received_for_free INTEGER
written_during_early_access INTEGER
                review_text    TEXT
               cleaned_text    TEXT

--- groupby 활용 예시 ---
시기별 리뷰 수 확인:
review_month  cnt  pos_rate
     2022-10  177     100.0
     2022-11  631      99.2
     2022-12  255      97.3
     2023-01  462      98.9
     2023-02  219      99.5
     2023-03  564      96.6
     2023-04  310      97.4
     2023-05  203      97.5
     2023-06  980      97.3
     2023-07 9280      97.1


---
## 9. DB 연결 종료

In [12]:
conn.close()
print('✅ DB 연결 종료')

✅ DB 연결 종료
